# Spatial Analysis

## Overview

Spatial analysis quantifies patterns in geographic data — are features clustered or dispersed? How does a variable change across space? What drives spatial variation in ecological outcomes?

**Core methods:**

| Method | Question | Output |
|---|---|---|
| Kernel density estimation | Where are features concentrated? | Density surface |
| Moran's I | Is there spatial autocorrelation? | I statistic, p-value |
| Semivariogram | How does similarity decay with distance? | Range, sill, nugget |
| Hotspot analysis | Which locations are significant clusters? | Z-score map |
| Spatial regression | Does accounting for space change coefficients? | Spatially-adjusted model |

**Tobler's first law of geography:** everything is related to everything else, but near things are more related than distant things — spatial autocorrelation is the rule, not the exception.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
from scipy.spatial.distance import cdist
from scipy.stats import norm
import warnings; warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)

# Generate spatially autocorrelated monitoring data
# Use Gaussian process to simulate realistic spatial structure
def simulate_spatial_field(n, range_m=3000, nugget=0.1, sill=1.0, seed=42):
    r = np.random.default_rng(seed)
    # Random locations in BNG
    x = r.uniform(317000, 327000, n)
    y = r.uniform(731000, 741000, n)
    coords = np.column_stack([x, y])
    # Exponential covariance
    D  = cdist(coords, coords)
    C  = sill * np.exp(-D / range_m) + nugget * np.eye(n)
    # Cholesky sample
    L  = np.linalg.cholesky(C + 1e-6*np.eye(n))
    z  = L @ r.standard_normal(n)
    return x, y, z

n_sites = 80
x, y, z_spatial = simulate_spatial_field(n_sites, range_m=2500)
# Convert to richness-like variable
richness = (10 + 8 * (z_spatial - z_spatial.min()) / (z_spatial.ptp() + 1e-8)).round(1)
gdf = gpd.GeoDataFrame({'richness': richness},
    geometry=[Point(xi, yi) for xi, yi in zip(x, y)], crs='EPSG:27700')
print(f"Simulated {n_sites} sites, richness: {richness.min():.1f}–{richness.max():.1f}")

---
## Kernel Density Estimation

In [ ]:
from scipy.stats import gaussian_kde

# KDE surface for monitoring site density
coords_kde = np.vstack([x, y])
kde = gaussian_kde(coords_kde, bw_method=0.2)
# Evaluate on regular grid
grid_x = np.linspace(317000, 327000, 100)
grid_y = np.linspace(731000, 741000, 100)
GX, GY = np.meshgrid(grid_x, grid_y)
Z_kde  = kde(np.vstack([GX.ravel(), GY.ravel()])).reshape(GX.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].contourf(GX, GY, Z_kde, levels=15, cmap='YlOrRd', alpha=0.8)
axes[0].scatter(x, y, c='black', s=20, zorder=2, alpha=0.6)
axes[0].set_title('Kernel Density Estimation
(monitoring site density)')
axes[0].set_xlabel('Easting (m)'); axes[0].set_ylabel('Northing (m)')

# Richness surface using IDW (Inverse Distance Weighting)
def idw_predict(x, y, z, grid_x, grid_y, power=2):
    gx_flat = GX.ravel(); gy_flat = GY.ravel()
    z_pred  = np.zeros(len(gx_flat))
    for i, (px, py) in enumerate(zip(gx_flat, gy_flat)):
        dists = np.sqrt((x-px)**2 + (y-py)**2)
        dists = np.maximum(dists, 1.0)
        weights = 1 / dists**power
        z_pred[i] = (weights * z).sum() / weights.sum()
    return z_pred.reshape(GX.shape)

Z_idw = idw_predict(x, y, richness, GX, GY)
im = axes[1].contourf(GX, GY, Z_idw, levels=15, cmap='YlGn', alpha=0.85)
axes[1].scatter(x, y, c=richness, cmap='YlGn', s=40, edgecolors='black',
                linewidths=0.5, zorder=2)
plt.colorbar(im, ax=axes[1], label='Species richness', shrink=0.8)
axes[1].set_title("IDW Interpolation
(species richness surface)")
axes[1].set_xlabel('Easting (m)')
plt.tight_layout(); plt.show()

---
## Moran's I: Spatial Autocorrelation

In [ ]:
# Moran's I: measures spatial autocorrelation
# I > 0: clustered; I < 0: dispersed; I ≈ 0: random
def morans_i(z, coords, k=8):
    n = len(z)
    z_bar = z.mean()
    z_dev = z - z_bar
    # Build binary k-nearest-neighbour weight matrix
    D  = cdist(coords, coords)
    W  = np.zeros((n, n))
    for i in range(n):
        nn_idx = np.argsort(D[i])[1:k+1]
        W[i, nn_idx] = 1
    W /= W.sum()   # row-normalise
    # Moran's I
    numerator   = n * (W * np.outer(z_dev, z_dev)).sum()
    denominator = (z_dev**2).sum()
    I = numerator / denominator
    # Permutation test for significance
    n_perm = 999
    I_perm = []
    for _ in range(n_perm):
        z_shuf = rng.permutation(z)
        z_d    = z_shuf - z_shuf.mean()
        I_perm.append(n * (W * np.outer(z_d, z_d)).sum() / (z_d**2).sum())
    p_val = (np.abs(I_perm) >= np.abs(I)).mean()
    return I, np.array(I_perm), p_val

coords_arr = np.column_stack([x, y])
I, I_perm, p = morans_i(richness, coords_arr, k=8)
print(f"Moran's I = {I:.4f}  (permutation p = {p:.4f})")
print(f"  I > 0: positive spatial autocorrelation (clusters of similar richness)")
print(f"  Expected I under H0: {-1/(n_sites-1):.4f}")
fig, ax = plt.subplots(figsize=(8,3))
ax.hist(I_perm, bins=40, color='steelblue', alpha=0.7, edgecolor='white', density=True)
ax.axvline(I, color='#e74c3c', lw=2.5, label=f"Observed I={I:.3f}")
ax.set_xlabel("Moran's I"); ax.set_ylabel('Density')
ax.set_title("Moran's I Permutation Test"); ax.legend()
plt.tight_layout(); plt.show()

---
## Semivariogram Analysis

In [ ]:
# Empirical semivariogram: how does variance grow with distance?
def empirical_semivariogram(z, coords, n_bins=12, max_dist=None):
    n   = len(z)
    D   = cdist(coords, coords)
    if max_dist is None:
        max_dist = D.max() * 0.5   # rule of thumb: max lag = half max dist
    bins     = np.linspace(0, max_dist, n_bins+1)
    gamma, h = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (D > lo) & (D <= hi) & (D > 0)
        if mask.sum() > 0:
            pairs = []
            for i, j in zip(*np.where(mask)):
                if i < j:
                    pairs.append((z[i] - z[j])**2)
            if pairs:
                gamma.append(np.mean(pairs) / 2)
                h.append((lo+hi)/2)
    return np.array(h), np.array(gamma)

h, gamma = empirical_semivariogram(richness, coords_arr, n_bins=14)
# Fit exponential model: gamma(h) = nugget + (sill-nugget)*(1-exp(-h/range))
from scipy.optimize import curve_fit
def exp_model(h, nugget, sill, range_m):
    return nugget + (sill - nugget) * (1 - np.exp(-h / range_m))
try:
    popt, _ = curve_fit(exp_model, h, gamma, p0=[0.5, 5.0, 2000], maxfev=5000)
    nugget_f, sill_f, range_f = popt
    h_fit = np.linspace(0, h.max(), 200)
    gamma_fit = exp_model(h_fit, *popt)
    fitted = True
except Exception:
    fitted = False
fig, ax = plt.subplots(figsize=(9,4))
ax.plot(h/1000, gamma, 'o', color='steelblue', ms=8, label='Empirical')
if fitted:
    ax.plot(h_fit/1000, gamma_fit, '-', color='#e74c3c', lw=2,
            label=f'Exp model: nugget={nugget_f:.2f}, sill={sill_f:.2f}, range={range_f/1000:.1f}km')
ax.set_xlabel('Lag distance (km)'); ax.set_ylabel('Semivariance γ(h)')
ax.set_title('Empirical Semivariogram with Exponential Fit'); ax.legend()
plt.tight_layout(); plt.show()
if fitted:
    print(f"Nugget: {nugget_f:.3f}  (micro-scale variance / measurement error)")
    print(f"Sill:   {sill_f:.3f}  (total variance at large lags)")
    print(f"Range:  {range_f/1000:.2f} km  (spatial autocorrelation disappears beyond this)")

In [ ]:
# Local Moran's I (LISA): identify spatial clusters and outliers
def local_morans(z, coords, k=8):
    n    = len(z)
    z_bar = z.mean()
    s2   = ((z - z_bar)**2).sum() / n
    D    = cdist(coords, coords)
    W    = np.zeros((n, n))
    for i in range(n):
        nn_idx = np.argsort(D[i])[1:k+1]
        W[i, nn_idx] = 1
    W /= W.sum()
    z_std = (z - z_bar) / np.sqrt(s2 + 1e-8)
    Ii    = z_std * (W @ z_std)
    return Ii

Ii = local_morans(richness, coords_arr)
# Classify: HH, LL (clusters), HL, LH (outliers)
z_std = (richness - richness.mean()) / (richness.std() + 1e-8)
Wz    = np.zeros(n_sites)
for i in range(n_sites):
    dists = cdist(coords_arr[i:i+1], coords_arr)[0]
    nn    = np.argsort(dists)[1:9]
    Wz[i] = richness[nn].mean()
Wz_std = (Wz - Wz.mean()) / (Wz.std() + 1e-8)
cluster_type = []
for zs, wzs in zip(z_std, Wz_std):
    if   zs > 0 and wzs > 0: cluster_type.append('HH')
    elif zs < 0 and wzs < 0: cluster_type.append('LL')
    elif zs > 0 and wzs < 0: cluster_type.append('HL')
    else:                      cluster_type.append('LH')
color_map = {'HH':'#e74c3c','LL':'steelblue','HL':'#e67e22','LH':'#9b59b6'}
colors = [color_map[c] for c in cluster_type]
fig, ax = plt.subplots(figsize=(8,7))
sc = ax.scatter(x, y, c=colors, s=[abs(I_i)*500+50 for I_i in Ii],
                edgecolors='white', linewidths=0.5)
from matplotlib.patches import Patch
legend_els = [Patch(color=v, label=k) for k,v in color_map.items()]
ax.legend(handles=legend_els, title='LISA cluster', loc='lower right')
ax.set_title('Local Moran's I (LISA)
Size = |Ii|, Colour = cluster type')
ax.set_xlabel('Easting (m)'); ax.set_ylabel('Northing (m)')
plt.tight_layout(); plt.show()
from collections import Counter
print("LISA cluster type counts:", dict(Counter(cluster_type)))

---

## Common Pitfalls

**1. Ignoring spatial autocorrelation in regression models**  
Standard OLS regression assumes independent residuals. When residuals are spatially autocorrelated, standard errors are underestimated and p-values are anti-conservative — false discovery rates can be very high. Always test residuals for spatial autocorrelation with Moran's I and use spatial regression (spatial lag or spatial error model) if significant.

**2. Using too few bins or too large a maximum lag in the semivariogram**  
With too few bins, the semivariogram is too coarse to identify the range. With maximum lag beyond half the study extent, each lag bin contains few pairs and the estimate is unreliable. Follow the rule of thumb: max lag ≤ half the maximum inter-point distance, with 10–20 lag bins.

**3. Interpreting global Moran's I without examining local clustering patterns**  
A near-zero global Moran's I may result from cancelling positive and negative local clusters. Always compute LISA (Local Indicators of Spatial Association) to identify where clusters and spatial outliers occur — the global statistic can obscure important local heterogeneity.

**4. Applying KDE bandwidth without considering the scale of the process**  
The bandwidth parameter controls the degree of smoothing in KDE. A bandwidth that is too small produces a spiky surface that overfits sample locations; too large erases real spatial variation. Choose bandwidth based on the expected scale of the underlying ecological process, and always inspect the surface at multiple bandwidths.

**5. Confusing the semivariogram range with ecological habitat scale**  
The fitted range from a variogram describes the scale at which the variable becomes spatially uncorrelated — but it depends on the specific variable, sampling scheme, and resolution. Do not extrapolate the range to claim general statements about habitat scale without considering these dependencies.
---
*python_methods_library - Samantha McGarrigle*